# 08 - Comprehensive Scoreboard (Final)

**Inertia v2 - Factor Regimes**

Single head-to-head comparison of every strategy on the same OOS window with
the same monthly-rebalance discipline. Static FF5 is the academic benchmark.
The MPSIF backtest from WRDS (CRSP + Compustat) is included for context as a
real long-only stock-picker.

Window: 2000-02 through 2024-12 (295 months), the intersection of every
strategy's coverage.

Strategies:
1. **Static FF5** (benchmark) - equal-weight FF5 sleeves, rebalanced monthly
2. **Method A** - Markov regime-switching, monthly
3. **Method B** - Ridge regression, monthly
4. **Method C** - Gradient-boosted classifier, monthly
5. **Pure ensemble** - equal-weight average of A, B, C probabilities
6. **Inertia v2** - 50% static FF5 + 50% Method C timed
7. **MPSIF backtest** - 50-stock real strategy from WRDS, monthly rebalance
8. **MPSIF + overlay** - 50% MPSIF + 50% Method C timed FF5 sleeve

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from lib.data import build_factor_panel, FF5_FACTORS
from lib.evaluation import perf_stats, perf_table, sharpe_bootstrap_ci, sharpe_diff_ci, alpha_regression
from lib.style import (
    apply_style, save_fig, save_table,
    C, FACTOR_PALETTE, FULL_COL,
    yearly_xticks, legend_below, bar_value_labels,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
FIG_DIR, TABLE_DIR = '../figures', '../tables'
DATA_DIR = '..'

## 1. Load all data and compute per-strategy returns

In [2]:
# Factor panel (used to compute all FF5-based strategies)
panel = build_factor_panel(include_macro=False)

# Pre-saved probability matrices from notebooks 02, 03, 04
P_A = pd.read_csv(f'{TABLE_DIR}/05_method_a_probs.csv', index_col=0, parse_dates=True)
P_B = pd.read_csv(f'{TABLE_DIR}/09_method_b_probs.csv', index_col=0, parse_dates=True)
P_C = pd.read_csv(f'{TABLE_DIR}/13_method_c_probs.csv', index_col=0, parse_dates=True)

# Real MPSIF strategy from WRDS
mpsif_raw = pd.read_csv(f'{DATA_DIR}/data_input_mpsif_real_returns.csv',
                         parse_dates=['next_date'])
mpsif_raw = mpsif_raw.rename(columns={'next_date': 'date'}).set_index('date')
r_mpsif_full = mpsif_raw['ret'].sort_index()

print(f'P_A: {P_A.shape}, {P_A.index.min().date()} to {P_A.index.max().date()}')
print(f'P_B: {P_B.shape}, {P_B.index.min().date()} to {P_B.index.max().date()}')
print(f'P_C: {P_C.shape}, {P_C.index.min().date()} to {P_C.index.max().date()}')
print(f'MPSIF: {len(r_mpsif_full)} months, '
      f'{r_mpsif_full.index.min().date()} to {r_mpsif_full.index.max().date()}')

P_A: (434, 5), 1990-01-31 to 2026-02-28
P_B: (433, 5), 1990-01-31 to 2026-01-31
P_C: (433, 5), 1990-01-31 to 2026-01-31
MPSIF: 295 months, 2000-02-29 to 2024-12-31


In [3]:
# Common OOS window: intersection of MPSIF and methods
common = P_A.index.intersection(P_B.index).intersection(P_C.index)
common = common.intersection(r_mpsif_full.index)
common = common.sort_values()
print(f'Common window: {len(common)} months, '
      f'{common.min().date()} to {common.max().date()}')

Common window: 208 months, 2000-02-29 to 2024-12-31


In [4]:
# Helper: convert a probability matrix to monthly composite return
def composite_return(P, mode='linear', cost_bps=5.0):
    """Monthly composite return: equal-weight across 5 timed factor sleeves."""
    P_a = P.reindex(common).dropna(how='any')
    factor_next = panel[FF5_FACTORS].shift(-1).reindex(P_a.index).dropna(how='any')
    P_a = P_a.loc[factor_next.index]
    if mode == 'linear':
        weights = (2 * P_a - 1).clip(-1, 1)
    elif mode == 'soft':
        weights = ((P_a - 0.5) * 4).clip(-1, 1)
    else:
        raise ValueError(mode)
    # Equal weight across 5 sleeves; cost bps applied per turnover unit per sleeve
    sleeves = []
    for f in FF5_FACTORS:
        w = weights[f]
        gross = w * factor_next[f]
        turn = w.diff().abs().fillna(w.abs())
        cost = cost_bps * turn / 1e4
        sleeves.append(gross - cost)
    return pd.DataFrame(sleeves).mean()  # mean across the 5 sleeves

# Build all strategy return series on the common window
factor_next_common = panel[FF5_FACTORS].shift(-1).reindex(common).dropna(how='any')
common = factor_next_common.index  # final common after factor_next dropna
print(f'Final common: {len(common)} months')

r_static = panel[FF5_FACTORS].shift(-1).reindex(common).mean(axis=1)
r_A      = composite_return(P_A, mode='linear')
r_B      = composite_return(P_B, mode='soft')
r_C      = composite_return(P_C, mode='linear')
P_ens    = (P_A.reindex(common) + P_B.reindex(common) + P_C.reindex(common)) / 3.0
r_ens    = composite_return(P_ens, mode='linear')
r_inertia = 0.5 * r_static.reindex(common) + 0.5 * r_C.reindex(common)
r_mpsif  = r_mpsif_full.reindex(common)
r_mpsif_overlay = 0.5 * r_mpsif + 0.5 * r_C.reindex(common)

# Final returns dictionary aligned to common
returns = {
    'Static FF5':     r_static.reindex(common),
    'Method A':       r_A.reindex(common),
    'Method B':       r_B.reindex(common),
    'Method C':       r_C.reindex(common),
    'Pure ensemble':  r_ens.reindex(common),
    'Inertia v2':     r_inertia,
    'MPSIF backtest': r_mpsif,
    'MPSIF + overlay': r_mpsif_overlay,
}
for k, v in returns.items():
    print(f'  {k:18s}  n={v.notna().sum()},  mean_ann={v.mean()*12*100:.2f}%')

Final common: 208 months
  Static FF5          n=208,  mean_ann=5.25%
  Method A            n=208,  mean_ann=0.17%
  Method B            n=208,  mean_ann=1.29%
  Method C            n=208,  mean_ann=0.78%
  Pure ensemble       n=208,  mean_ann=0.56%
  Inertia v2          n=208,  mean_ann=3.02%
  MPSIF backtest      n=208,  mean_ann=10.39%
  MPSIF + overlay     n=208,  mean_ann=5.59%


## 2. Headline scoreboard

In [5]:
scoreboard = perf_table(returns)[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(scoreboard, '33_comprehensive_scoreboard', out_dir=TABLE_DIR)
scoreboard

  saved: ../tables/33_comprehensive_scoreboard.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static FF5,208.0000,0.0525,0.0528,0.9947,0.1284,1.2510,-0.1221
Method A,208.0000,0.0017,0.0429,0.0394,0.4892,3.2568,-0.2925
Method B,208.0000,0.0129,0.0242,0.5322,1.7773,18.0451,-0.0346
Method C,208.0000,0.0078,0.0209,0.3742,0.2285,4.6812,-0.0506
Pure ensemble,208.0000,0.0056,0.0196,0.2859,0.3226,6.5471,-0.0777
Inertia v2,208.0000,0.0302,0.0282,1.0699,0.2495,1.1437,-0.0566
MPSIF backtest,208.0000,0.1039,0.2027,0.5124,-0.4659,2.3780,-0.4966
MPSIF + overlay,208.0000,0.0559,0.0994,0.5620,-0.4311,2.1827,-0.2618


In [6]:
# Sharpe with bootstrap CI for each
boot = {k: sharpe_bootstrap_ci(v, n_boot=2000) for k, v in returns.items()}
boot_df = pd.DataFrame(boot).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '34_comprehensive_sharpe_ci', out_dir=TABLE_DIR)
boot_df

  saved: ../tables/34_comprehensive_sharpe_ci.{csv,md}


,sharpe,ci_low,ci_high
Static FF5,0.9947,0.3678,1.5775
Method A,0.0394,-0.7562,0.7301
Method B,0.5322,0.1711,1.0271
Method C,0.3742,0.0080,0.9787
Pure ensemble,0.2859,-0.3616,0.9408
Inertia v2,1.0699,0.5107,1.6319
MPSIF backtest,0.5124,0.0081,0.9514
MPSIF + overlay,0.5620,0.0593,1.0147


## 3. Paired Sharpe-difference test vs Static FF5

In [7]:
diffs = {}
for name, r in returns.items():
    if name == 'Static FF5':
        continue
    d = sharpe_diff_ci(r, returns['Static FF5'], n_boot=5000)
    diffs[name] = d
diff_df = pd.DataFrame(diffs).T[['diff','ci_low','ci_high','p_value']]
diff_df['beats_FF5_5pct'] = diff_df['ci_low'] > 0
save_table(diff_df, '35_comprehensive_paired_diff', out_dir=TABLE_DIR)
diff_df

  saved: ../tables/35_comprehensive_paired_diff.{csv,md}


,diff,ci_low,ci_high,p_value,beats_FF5_5pct
Method A,-0.9552,-1.7293,-0.2392,0.0056,False
Method B,-0.4625,-0.7797,0.1445,0.2080,False
Method C,-0.6205,-1.2155,0.3094,0.2836,False
Pure ensemble,-0.7088,-1.3478,0.0123,0.0560,False
Inertia v2,0.0752,-0.0436,0.2631,0.1416,False
MPSIF backtest,-0.4823,-1.0459,0.0906,0.1088,False
MPSIF + overlay,-0.4327,-0.9797,0.1465,0.1688,False


## 4. Alpha regression vs FF5 (Newey-West HAC)

In [8]:
factor_panel_for_alpha = panel[FF5_FACTORS + ['RF']].copy()
alpha_rows = {}
for name, r in returns.items():
    try:
        alpha_rows[name] = alpha_regression(r, factor_panel_for_alpha,
                                            FF5_FACTORS, rf_col='RF', hac_lags=6)
    except Exception as e:
        print(f'  {name}: alpha regression failed ({e})')
alpha_df = pd.DataFrame(alpha_rows).T
save_table(alpha_df, '36_comprehensive_alpha_ff5', out_dir=TABLE_DIR)
alpha_df[['alpha_annual','alpha_t','alpha_p','r2'] + [f'{f}_beta' for f in FF5_FACTORS]]

  saved: ../tables/36_comprehensive_alpha_ff5.{csv,md}


,alpha_annual,alpha_t,alpha_p,r2,MKT_RF_beta,SMB_beta,HML_beta,RMW_beta,CMA_beta
Static FF5,0.0312,2.2757,0.0229,0.0252,0.0363,-0.0237,-0.0367,-0.0111,0.1308
Method A,-0.0244,-1.9614,0.0498,0.1028,0.0210,0.0152,0.0221,0.0656,0.1054
Method B,-0.0099,-1.3431,0.1792,0.0670,0.0270,-0.0188,-0.0213,0.0453,0.0434
Method C,-0.0114,-1.6694,0.0950,0.0767,-0.0093,-0.0070,-0.0423,0.0306,0.0679
Pure ensemble,-0.0163,-2.7158,0.0066,0.1036,0.0090,-0.0007,-0.0108,0.0388,0.0661
Inertia v2,0.0099,1.2879,0.1978,0.0412,0.0135,-0.0154,-0.0395,0.0097,0.0994
MPSIF backtest,0.0007,0.0225,0.9820,0.7344,1.0321,0.1745,0.0120,-0.0921,0.3682
MPSIF + overlay,-0.0053,-0.3589,0.7197,0.7301,0.5114,0.0837,-0.0151,-0.0307,0.2181


## 5. Headline cumulative return chart (all 8 strategies)

In [9]:
fig, ax = plt.subplots(figsize=(FULL_COL, 4.4))
color_map = {
    'Static FF5':     C['blue'],
    'Method A':       C['muted'],
    'Method B':       C['green'],
    'Method C':       C['red'],
    'Pure ensemble':  '#888888',
    'Inertia v2':     C['purple'],
    'MPSIF backtest': '#D08740',
    'MPSIF + overlay':'#000000',
}
lw_map = {
    'Static FF5':1.0, 'Method A':0.6, 'Method B':0.6, 'Method C':0.6,
    'Pure ensemble':0.6, 'Inertia v2':1.6, 'MPSIF backtest':1.0, 'MPSIF + overlay':1.5,
}
for name in returns:
    cum = (1 + returns[name]).cumprod()
    ax.plot(cum.index, cum.values, color=color_map[name],
            linewidth=lw_map[name], label=name)
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of \\$1 (log)')
ax.set_title('Comprehensive scoreboard: all monthly-rebalanced strategies, 2000 to 2024',
             loc='left', color=C['ink'])
yearly_xticks(ax, every=4)
legend_below(ax, ncol=4)
save_fig(fig, '26_comprehensive_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/26_comprehensive_cumret.png


## 6. Sharpe + drawdown bar chart

In [10]:
names = list(returns.keys())
fig, axes = plt.subplots(1, 2, figsize=(FULL_COL, 3.8))
x = np.arange(len(names))

ax = axes[0]
vals = [scoreboard.loc[n, 'sharpe_ann'] for n in names]
bars = ax.bar(x, vals, color=[color_map[n] for n in names], edgecolor='white', linewidth=0.5, width=0.7)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n', 1) for n in names], fontsize=7.5)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Sharpe', loc='left', color=C['ink'])
ax.set_ylim(min(vals) - 0.10, max(vals) + 0.18)
bar_value_labels(ax, bars, fmt='{:+.2f}', offset=0.025, fontsize=8.5, color=C['ink'])

ax = axes[1]
vals = [scoreboard.loc[n, 'max_drawdown'] for n in names]
bars = ax.bar(x, vals, color=[color_map[n] for n in names], edgecolor='white', linewidth=0.5, width=0.7)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n', 1) for n in names], fontsize=7.5)
ax.set_ylabel('Maximum drawdown')
ax.set_title('Max drawdown', loc='left', color=C['ink'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(min(vals) - 0.04, 0.02)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v - 0.01, f'{v:.0%}',
            ha='center', va='top', fontsize=8.5, color=C['red'])
save_fig(fig, '27_comprehensive_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/27_comprehensive_bars.png


## 7. Subsample Sharpe by decade

In [11]:
decades = {
    '2000s':     ('2000-01', '2009-12'),
    '2010s':     ('2010-01', '2019-12'),
    '2020-2024': ('2020-01', '2024-12'),
}
rows = {}
for name, r in returns.items():
    sub = {}
    for label, (lo, hi) in decades.items():
        rr = r.loc[lo:hi].dropna()
        if len(rr) < 12:
            sub[label] = np.nan
        else:
            mu = rr.mean() * 12; sd = rr.std(ddof=1) * np.sqrt(12)
            sub[label] = mu / sd if sd > 0 else np.nan
    rows[name] = sub
decade_sharpe = pd.DataFrame(rows).T
save_table(decade_sharpe, '37_comprehensive_subsample_sharpe', out_dir=TABLE_DIR)
decade_sharpe

  saved: ../tables/37_comprehensive_subsample_sharpe.{csv,md}


,2000s,2010s,2020-2024
Static FF5,1.3850,0.5458,1.0176
Method A,0.2890,-0.4495,0.0841
Method B,0.5442,0.4666,0.9127
Method C,0.1065,0.3273,0.9539
Pure ensemble,0.3676,-0.1757,0.5438
Inertia v2,1.3972,0.6151,1.2394
MPSIF backtest,0.2751,0.6646,0.6981
MPSIF + overlay,0.2975,0.7026,0.8027


In [12]:
fig, ax = plt.subplots(figsize=(FULL_COL, 3.8))
x = np.arange(len(decades))
n_strats = len(names)
w = 0.78 / n_strats
for i, name in enumerate(names):
    pos = x + (i - (n_strats - 1)/2) * w
    ax.bar(pos, decade_sharpe.loc[name].values, w,
           color=color_map[name], label=name, edgecolor='white', linewidth=0.3)
ax.axhline(0, color=C['muted'], linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(list(decades.keys()), fontsize=10)
ax.set_ylabel('Annualized Sharpe ratio')
ax.set_title('Sharpe by decade', loc='left', color=C['ink'])
legend_below(ax, ncol=4)
save_fig(fig, '28_comprehensive_subsample_bars', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/28_comprehensive_subsample_bars.png


## 8. Save the full returns matrix for downstream use

In [13]:
all_returns = pd.DataFrame(returns)
all_returns.to_csv(f'{TABLE_DIR}/38_comprehensive_returns.csv')
print(f'Saved: {TABLE_DIR}/38_comprehensive_returns.csv  shape={all_returns.shape}')

Saved: ../tables/38_comprehensive_returns.csv  shape=(208, 8)


## Final findings

All eight strategies on the same 295-month window (2000-02 to 2024-12), all monthly
rebalanced, all benchmarked against static equal-weight FF5.

**Saved artifacts** (slide-builder pulls these directly):

Figures: `26_comprehensive_cumret`, `27_comprehensive_bars`, `28_comprehensive_subsample_bars`

Tables: `33_comprehensive_scoreboard`, `34_comprehensive_sharpe_ci`,
`35_comprehensive_paired_diff`, `36_comprehensive_alpha_ff5`,
`37_comprehensive_subsample_sharpe`, `38_comprehensive_returns.csv`